In [ ]:
import os
from ebooklib import epub

# Caminho do seu e-book
CAMINHO_EPUB = "../assets/Crossroads of Twilight.epub"
ARQUIVO_SAIDA = "ebook_bruto_total.txt"

# Carrega o livro usando a biblioteca ebooklib
livro = epub.read_epub(CAMINHO_EPUB)

print(f"Iniciando extração bruta de: {os.path.basename(CAMINHO_EPUB)}")

# Abre o arquivo de saída em modo de escrita de texto com codificação UTF-8
with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as f_saida:
    
    # Percorre absolutamente todos os itens armazenados dentro do arquivo EPUB
    for indice, item in enumerate(livro.get_items(), start=1):
        
        # FILTRO CORRIGIDO: Verifica se o item é um objeto de HTML/Texto do e-book
        # Isso substitui o item.get_type() == epub.ITEM_DOCUMENT de forma segura
        if isinstance(item, epub.EpubHtml):
            
            # Pega os bytes puros do arquivo interno do e-book
            conteudo_binario = item.get_content()
            
            # Decodifica os bytes para string UTF-8 (exatamente como está gravado no arquivo)
            conteudo_texto_puro = conteudo_binario.decode('utf-8', errors='ignore')
            
            # Escreve um marcador visual apenas para você saber onde muda de arquivo interno
            f_saida.write(f"\n\n<!-- ==================== INÍCIO DO ITEM {indice}: {item.get_name()} ==================== -->\n\n")
            f_saida.write(conteudo_texto_puro)
            f_saida.write(f"\n\n<!-- ==================== FIM DO ITEM {indice}: {item.get_name()} ==================== -->\n")

print(f"Concluído! Todo o código e estrutura do e-book foram salvos em: {ARQUIVO_SAIDA}")

Iniciando extração bruta de: Crossroads of Twilight.epub
Concluído! Todo o código e estrutura do e-book foram salvos em: ebook_bruto_total.txt


In [63]:
import os
import json
from bs4 import BeautifulSoup
from ebooklib import epub

# Configurações de Caminho
CAMINHO_EPUB = "./assets/Crossroads of Twilight.epub"
ARQUIVO_JSON_SAIDA = "estrutura_livro.json"

livro = epub.read_epub(CAMINHO_EPUB)

dados_livro = {
    "titulo_arquivo": os.path.basename(CAMINHO_EPUB),
    "capitulos": []
}

contador_global = 1
indice_cap = 1

for item in livro.get_items():
    if isinstance(item, epub.EpubHtml):
        html_bruto = item.get_content()
        soup = BeautifulSoup(html_bruto, 'html.parser')
        
        # =================================================================
        # ETAPA DE NORMALIZAÇÃO: Converte spans de estilo em tags universais
        # =================================================================
        for span in soup.find_all('span'):
            classes = span.get('class', [])
            # Une as classes em uma string só para facilitar a busca (ex: "calibre_4 italic")
            classes_str = " ".join(classes).lower()
            
            # Verifica se a classe sugere itálico ou negrito
            e_italico = any(termo in classes_str for termo in ['it', 'ital', 'oblique'])
            e_negrito = any(termo in classes_str for termo in ['bold', 'bd', 'emp'])
            
            if e_italico and e_negrito:
                span.name = 'b'
                # Se for ambos, envolvemos o conteúdo interno em uma tag 'i'
                span.wrap(soup.new_tag('i'))
            elif e_italico:
                span.name = 'i'
            elif e_negrito:
                span.name = 'b'
                
            # Remove o atributo class para limpar o HTML que vai para a tradução
            if e_italico or e_negrito:
                del span['class']
        # =================================================================

        # --- Identificação Inteligente do Título do Capítulo ---
        titulo_capitulo = f"Capítulo {indice_cap}"
        if soup.title and soup.title.string:
            titulo_capitulo = soup.title.string.strip()
        elif soup.find(['h1', 'h2', 'h3']):
            titulo_capitulo = soup.find(['h1', 'h2', 'h3']).get_text().strip()
            
        if not titulo_capitulo or titulo_capitulo.lower() == "untitled":
            titulo_capitulo = f"Conteúdo {indice_cap}"

        dados_capitulo = {
            "id_capitulo": indice_cap,
            "nome_arquivo_interno": item.get_name(),
            "titulo_capitulo": titulo_capitulo,
            "conteudo": []
        }

        # --- Varredura Hierárquica Controlada ---
        for tag in soup.find_all(['p', 'h1', 'h2', 'h3', 'td', 'blockquote', 'img', 'image']):
            
            if tag.find_parent(['p', 'h1', 'h2', 'h3', 'td', 'blockquote']):
                continue

            # PROCESSAMENTO DE TEXTO
            if tag.name in ['p', 'h1', 'h2', 'h3', 'td', 'blockquote']:
                # Agora o decode_contents incluirá tags <i> e <b> limpas e universais
                texto_html = tag.decode_contents().strip()
                
                if not texto_html or texto_html in ["&nbsp;", "<br/>", "<br>"]:
                    continue
                
                dados_capitulo["conteudo"].append({
                    "id": contador_global,
                    "tipo": "texto",
                    "tag_original": tag.name,
                    "original": texto_html,
                    "traduzido": ""
                })
                contador_global += 1

            # PROCESSAMENTO DE IMAGENS
            elif tag.name in ['img', 'image']:
                src_imagem = tag.get('src') or tag.get('xlink:href') or tag.get('href')
                if not src_imagem:
                    continue

                dados_capitulo["conteudo"].append({
                    "id": contador_global,
                    "tipo": "imagem",
                    "tag_original": tag.name,
                    "src_original": src_imagem
                })
                contador_global += 1

        if dados_capitulo["conteudo"]:
            dados_livro["capitulos"].append(dados_capitulo)
            indice_cap += 1

with open(ARQUIVO_JSON_SAIDA, "w", encoding="utf-8") as f:
    json.dump(dados_livro, f, ensure_ascii=False, indent=4)

print(f"Varredura completa concluída com sucesso!")
print(f"Total de {indice_cap - 1} seções salvas em '{ARQUIVO_JSON_SAIDA}' contendo {contador_global - 1} elementos.")

Varredura completa concluída com sucesso!
Total de 48 seções salvas em 'estrutura_livro.json' contendo 3212 elementos.
